# Lab 26-01 · Budgets, Flow-Down and Decisions

Companion to [26-01](../../chapters/26-01-requirements-and-specifications.md). Each part starts with a prediction: write yours down before running the cell.

In [1]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import norm
import reqs as r

## Part 1 · Does the distribution matter?

*Predict:* if each of Example 26-01.3's contributions is uniform with its ±bound as the half-width, does the out-of-spec fraction rise, fall or stay near 66 ppm?

In [2]:
b = [0.40, 0.35, 0.30, 0.25, 0.20, 0.30]
g = r.mc_budget(b, dist="normal", n=2_000_000)
u = r.mc_budget(b, dist="uniform", n=2_000_000)
sig_u = np.sqrt(sum(x**2 / 3 for x in b))
print(f"Gaussian bounds at 3 sigma: out of spec {1e6 * (abs(g) > 1).mean():.0f} ppm (closed form {1e6 * r.p_outside(r.stack_rss(b) / 3, 1.0):.0f})")
print(f"Uniform bounds:             out of spec {100 * (abs(u) > 1).mean():.2f} %, sigma {sig_u:.3f} %")
print(f"Gaussian with the same sigma: {100 * r.p_outside(sig_u, 1.0):.2f} %")
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.hist(g, bins=200, density=True, histtype="step", label="Gaussian contributions")
ax.hist(u, bins=200, density=True, histtype="step", label="uniform contributions")
for s in (-1, 1):
    ax.axvline(s * 1.0, color="k", lw=1)
ax.set_xlabel("total error (% of full scale)"); ax.set_ylabel("density"); ax.legend()
plt.show()

Gaussian bounds at 3 sigma: out of spec 64 ppm (closed form 66)
Uniform bounds:             out of spec 1.77 %, sigma 0.434 %
Gaussian with the same sigma: 2.12 %


The same "±bound" means a σ of $b/3$ for a 3σ Gaussian but $b/\sqrt3$ for a uniform. State the distribution and the confidence level with every bound.

## Part 2 · Display time

*Predict:* the display duty that gives exactly 168 h with no margin.

In [3]:
def life(d):
    prof = [(0.82 - d, 0.050), (0.15, 2.0), (0.03, 10.0), (d, 25.0)]
    return r.battery_life_h(300, 0.85, r.duty_average(prof))

duty = np.linspace(0, 0.06, 121)
hours = np.array([life(d) for d in duty])
from scipy.optimize import brentq
d168 = brentq(lambda d: life(d) - 168, 0, 0.06)
print(f"display duty for 168 h: {100 * d168:.2f} % ({1440 * d168:.1f} min a day)")
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(1440 * duty, hours)
ax.axhline(168, color="k", lw=1, ls="--")
ax.set_xlabel("display on-time (min per day)"); ax.set_ylabel("battery life (h)")
plt.show()

display duty for 168 h: 3.51 % (50.6 min a day)


## Part 3 · How robust is the review's value?

*Predict:* the range of Example 26-01.5's saving when the escalation factors are drawn from their NASA ranges.

In [4]:
ranges = [(3, 8), (7, 16), (21, 78), (29, 1500)]
down = [0.4, 0.3, 0.6]

def saving(f):
    return r.defect_flow(100, [0.2] + down, f)[1] - r.defect_flow(100, [0.6] + down, f)[1]

lo = saving([1] + [a for a, _ in ranges]); hi = saving([1] + [b_ for _, b_ in ranges])
print(f"saving at the low ends {lo:.0f} units, at the high ends {hi:.0f} units")
rng = np.random.default_rng(1)
s = []
for _ in range(10_000):
    f = [1] + [float(np.exp(rng.uniform(np.log(a), np.log(b_)))) for a, b_ in ranges]
    s.append(saving(f))
s = np.array(s)
print("5th, 50th, 95th percentiles:", np.percentile(s, [5, 50, 95]).round(0))
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.hist(s, bins=60)
ax.set_xlabel("saving (relative cost units)"); ax.set_ylabel("count")
plt.show()

saving at the low ends 465 units, at the high ends 11069 units


5th, 50th, 95th percentiles: [ 709. 1949. 8796.]


## Part 4 · Decision sweep

*Predict:* the capability weight at which B overtakes A.

In [5]:
S = np.array([[5, 2, 5, 5], [4, 4, 3, 3], [3, 5, 2, 2]], dtype=float)
w = [0.40, 0.25, 0.20, 0.15]
ws = np.linspace(0, 1, 201)
sc = r.scores_vs_weight(w, S, 1, ws)
for a, b_, name in ((0, 1, "A-B"), (0, 2, "A-C"), (1, 2, "B-C")):
    print(f"{name} tie at capability weight {r.flip_weight(w, S, 1, a, b_):.3f}")
fig, ax = plt.subplots(figsize=(6, 3.5))
for k, name in enumerate(["A: MCU", "B: MCU + DSP", "C: MCU + NPU"]):
    ax.plot(ws, sc[:, k], label=name)
ax.axvline(0.25, color="k", lw=1, ls=":")
ax.set_xlabel("capability weight"); ax.set_ylabel("weighted score"); ax.legend()
plt.show()

A-B tie at capability weight 0.423
A-C tie at capability weight 0.451
B-C tie at capability weight 0.500
